In [1]:
# =============================================================================
# TRACE THE ACE — FINAL MODEL VALIDATION AUDIT
# CELL 9 — STRUCTURAL LEAKAGE DECISION RESOLUTION
# =============================================================================

from pathlib import Path
import json
import re
import pandas as pd
import numpy as np


print("=" * 100)
print(
    "TRACE THE ACE — FINAL MODEL VALIDATION AUDIT"
)
print(
    "CELL 9 — STRUCTURAL LEAKAGE DECISION RESOLUTION"
)
print("=" * 100)


# =============================================================================
# 1. PROJECT / ARTIFACT CONTRACT
# =============================================================================

PROJECT_ROOT = Path(
    r"D:\Competition\Trace-the-race-local"
)

SCRATCH_ROOT = (
    PROJECT_ROOT
    / "scratch_mastery_outputs"
)

CELL6_ROOT = (
    SCRATCH_ROOT
    / "09C"
    / "final_model_validation_audit"
    / "cell6"
)

CELL9_ROOT = (
    SCRATCH_ROOT
    / "09C"
    / "final_model_validation_audit"
    / "cell9"
)

CELL9_ROOT.mkdir(
    parents=True,
    exist_ok=True
)


assert PROJECT_ROOT.exists(), (
    f"Project root not found:\n{PROJECT_ROOT}"
)

assert CELL6_ROOT.exists(), (
    f"Cell 6 artifact directory not found:\n{CELL6_ROOT}"
)


print("\n" + "=" * 100)
print("PROJECT / ARTIFACT CONTRACT")
print("=" * 100)

print(
    f"PROJECT_ROOT : {PROJECT_ROOT}"
)

print(
    f"CELL6_ROOT   : {CELL6_ROOT}"
)

print(
    f"CELL9_ROOT   : {CELL9_ROOT}"
)

print(
    "Project root : PASS"
)

print(
    "Cell 6 artifact root : PASS"
)


# =============================================================================
# 2. DISCOVER CELL 6 ARTIFACTS
# =============================================================================

json_files = sorted(
    CELL6_ROOT.rglob("*.json")
)

parquet_files = sorted(
    CELL6_ROOT.rglob("*.parquet")
)

csv_files = sorted(
    CELL6_ROOT.rglob("*.csv")
)


print("\n" + "=" * 100)
print("CELL 6 ARTIFACT DISCOVERY")
print("=" * 100)

print(
    f"JSON files     : {len(json_files)}"
)

for path in json_files:
    print(
        f"  JSON     : {path}"
    )

print(
    f"Parquet files : {len(parquet_files)}"
)

for path in parquet_files:
    print(
        f"  PARQUET  : {path}"
    )

print(
    f"CSV files     : {len(csv_files)}"
)

for path in csv_files:
    print(
        f"  CSV      : {path}"
    )


assert len(json_files) >= 1, (
    "No Cell 6 JSON artifact found."
)


# =============================================================================
# 3. LOAD STRUCTURAL LEAKAGE SUMMARY
# =============================================================================

preferred_json = [
    path
    for path in json_files
    if (
        "leakage" in path.name.lower()
        or
        "structural" in path.name.lower()
    )
]


if len(preferred_json) == 1:

    leakage_json_path = (
        preferred_json[0]
    )

elif len(preferred_json) > 1:

    # Prefer the exact artifact previously discovered.
    exact_candidates = [
        path
        for path in preferred_json
        if path.name.lower()
        ==
        "cell6_structural_leakage_summary.json"
    ]

    if len(exact_candidates) == 1:

        leakage_json_path = (
            exact_candidates[0]
        )

    else:

        raise AssertionError(
            "Multiple Cell 6 leakage JSON files found "
            "and no unique selection is possible:\n"
            +
            "\n".join(
                str(path)
                for path in preferred_json
            )
        )

else:

    raise AssertionError(
        "No structural/leakage JSON artifact found."
    )


with open(
    leakage_json_path,
    "r",
    encoding="utf-8",
) as f:

    leakage_payload = json.load(f)


assert isinstance(
    leakage_payload,
    dict,
), (
    "Cell 6 leakage JSON must contain "
    "a JSON object."
)


print("\n" + "=" * 100)
print("STRUCTURAL LEAKAGE SUMMARY")
print("=" * 100)

print(
    f"Artifact : {leakage_json_path}"
)

print(
    "JSON readable : PASS"
)

print(
    f"Top-level keys : "
    f"{sorted(leakage_payload.keys())}"
)


# =============================================================================
# 4. RECURSIVE FIELD INVENTORY
# =============================================================================

def flatten_json(
    obj,
    prefix="",
):
    """
    Recursively flatten a JSON object into
    dotted-path -> value mappings.
    """

    output = {}

    if isinstance(
        obj,
        dict,
    ):

        for key, value in obj.items():

            child_prefix = (
                f"{prefix}.{key}"
                if prefix
                else str(key)
            )

            output.update(
                flatten_json(
                    value,
                    child_prefix,
                )
            )

    elif isinstance(
        obj,
        list,
    ):

        # Preserve list itself as a value.
        output[prefix] = obj

        for idx, value in enumerate(obj):

            child_prefix = (
                f"{prefix}[{idx}]"
            )

            if isinstance(
                value,
                (dict, list),
            ):

                output.update(
                    flatten_json(
                        value,
                        child_prefix,
                    )
                )

    else:

        output[prefix] = obj

    return output


flat_payload = flatten_json(
    leakage_payload
)


print("\n" + "=" * 100)
print("LEAKAGE FIELD INVENTORY")
print("=" * 100)

print(
    f"Flattened fields : "
    f"{len(flat_payload)}"
)

for key, value in flat_payload.items():

    value_repr = repr(value)

    if len(value_repr) > 300:

        value_repr = (
            value_repr[:300]
            + " ..."
        )

    print(
        f"{key} = {value_repr}"
    )


# =============================================================================
# 5. IDENTIFY LEAKAGE-RELEVANT FIELDS
# =============================================================================

LEAKAGE_TERMS = [
    "leakage",
    "target",
    "label",
    "fold",
    "session",
    "response",
    "overlap",
    "contamination",
    "same_row",
    "same-row",
    "train",
    "valid",
    "validation",
    "oof",
    "future",
    "temporal",
]


relevant_fields = {
    key: value
    for key, value in flat_payload.items()
    if any(
        term in key.lower()
        for term in LEAKAGE_TERMS
    )
}


print("\n" + "=" * 100)
print("LEAKAGE-RELEVANT FIELDS")
print("=" * 100)

print(
    f"Relevant fields : "
    f"{len(relevant_fields)}"
)

for key, value in relevant_fields.items():

    value_repr = repr(value)

    if len(value_repr) > 300:

        value_repr = (
            value_repr[:300]
            + " ..."
        )

    print(
        f"{key} = {value_repr}"
    )


# =============================================================================
# 6. GENERIC BOOLEAN / DECISION EXTRACTION
# =============================================================================

def normalize_key(
    key,
):
    return re.sub(
        r"[^a-z0-9]+",
        "_",
        str(key).lower(),
    ).strip("_")


normalized_fields = {
    normalize_key(key): value
    for key, value in flat_payload.items()
}


def find_first_field(
    exact_names,
    contains_names=None,
):

    contains_names = (
        contains_names
        or []
    )

    # Exact normalized match first.
    for name in exact_names:

        name_norm = normalize_key(
            name
        )

        if name_norm in normalized_fields:

            return (
                name_norm,
                normalized_fields[
                    name_norm
                ],
            )

    # Then substring match.
    for key, value in normalized_fields.items():

        if any(
            term in key
            for term in contains_names
        ):

            return (
                key,
                value,
            )

    return (
        None,
        None,
    )


decision_key, decision_value = (
    find_first_field(
        exact_names=[
            "leakage_decision",
            "overall_leakage_decision",
            "structural_leakage_decision",
            "final_leakage_decision",
            "decision",
        ],
        contains_names=[
            "leakage_decision",
            "structural_leakage",
        ],
    )
)


direct_key, direct_value = (
    find_first_field(
        exact_names=[
            "direct_leakage_evidence",
            "leakage_detected",
            "leakage_present",
            "has_leakage",
        ],
        contains_names=[
            "direct_leakage",
            "leakage_detected",
            "leakage_present",
            "has_leakage",
        ],
    )
)


print("\n" + "=" * 100)
print("DECISION FIELD DISCOVERY")
print("=" * 100)

print(
    f"Decision field : "
    f"{decision_key}"
)

print(
    f"Decision value : "
    f"{decision_value!r}"
)

print(
    f"Direct leakage field : "
    f"{direct_key}"
)

print(
    f"Direct leakage value : "
    f"{direct_value!r}"
)


# =============================================================================
# 7. NUMERICAL SAFETY / COUNT AUDIT
# =============================================================================

numeric_findings = []

for key, value in relevant_fields.items():

    if isinstance(
        value,
        (int, float, np.integer, np.floating),
    ):

        if (
            not isinstance(
                value,
                bool,
            )
            and
            np.isfinite(
                float(value)
            )
        ):

            numeric_findings.append(
                {
                    "field": key,
                    "value": float(value),
                }
            )


print("\n" + "=" * 100)
print("NUMERICAL LEAKAGE / CONTAMINATION SIGNALS")
print("=" * 100)

if numeric_findings:

    for item in numeric_findings:

        print(
            f"{item['field']} : "
            f"{item['value']}"
        )

else:

    print(
        "No numerical leakage fields discovered."
    )


# =============================================================================
# 8. EXPLICIT POSITIVE LEAKAGE SIGNAL DETECTION
# =============================================================================

POSITIVE_TERMS = [
    "leakage_detected",
    "direct_leakage",
    "target_leakage",
    "label_leakage",
    "fold_leakage",
    "session_leakage",
    "response_leakage",
    "same_row_leakage",
    "contamination_detected",
]


positive_signals = []


for key, value in normalized_fields.items():

    key_has_positive_term = any(
        term in key
        for term in POSITIVE_TERMS
    )

    if not key_has_positive_term:
        continue


    if isinstance(
        value,
        bool,
    ):

        if value is True:

            positive_signals.append(
                {
                    "field": key,
                    "value": value,
                }
            )

    elif isinstance(
        value,
        (int, float),
    ):

        if float(value) > 0:

            positive_signals.append(
                {
                    "field": key,
                    "value": value,
                }
            )

    elif isinstance(
        value,
        str,
    ):

        text = value.lower().strip()

        if text in {
            "true",
            "yes",
            "detected",
            "present",
            "failed",
            "fail",
            "leakage_detected",
            "suspected",
        }:

            positive_signals.append(
                {
                    "field": key,
                    "value": value,
                }
            )


print("\n" + "=" * 100)
print("EXPLICIT POSITIVE LEAKAGE SIGNALS")
print("=" * 100)

if positive_signals:

    for item in positive_signals:

        print(
            f"  {item['field']} = "
            f"{item['value']!r}"
        )

else:

    print(
        "No explicit positive leakage signal found."
    )


# =============================================================================
# 9. EXPLICIT NEGATIVE LEAKAGE SIGNAL DETECTION
# =============================================================================

negative_signals = []


NEGATIVE_TERMS = [
    "no_leakage",
    "leakage_free",
    "leakage_detected",
    "direct_leakage_evidence",
    "target_leakage",
    "contamination",
]


for key, value in normalized_fields.items():

    if not any(
        term in key
        for term in NEGATIVE_TERMS
    ):
        continue


    if isinstance(
        value,
        bool,
    ):

        # For fields such as leakage_detected,
        # False is explicitly negative evidence.
        if value is False:

            negative_signals.append(
                {
                    "field": key,
                    "value": value,
                }
            )

    elif isinstance(
        value,
        (int, float),
    ):

        if (
            float(value) == 0
            and
            any(
                term in key
                for term in [
                    "count",
                    "overlap",
                    "contamination",
                    "leakage",
                ]
            )
        ):

            negative_signals.append(
                {
                    "field": key,
                    "value": value,
                }
            )

    elif isinstance(
        value,
        str,
    ):

        text = value.lower().strip()

        if text in {
            "false",
            "no",
            "none",
            "absent",
            "not_detected",
            "no_leakage",
            "leakage_free",
        }:

            negative_signals.append(
                {
                    "field": key,
                    "value": value,
                }
            )


print("\n" + "=" * 100)
print("EXPLICIT NEGATIVE LEAKAGE SIGNALS")
print("=" * 100)

if negative_signals:

    for item in negative_signals:

        print(
            f"  {item['field']} = "
            f"{item['value']!r}"
        )

else:

    print(
        "No explicit negative leakage signal found."
    )


# =============================================================================
# 10. DECISION RESOLUTION
# =============================================================================

#
# IMPORTANT:
#
# We never infer "NO LEAKAGE" merely because a field is missing.
#
# Decision hierarchy:
#
# 1. Explicit positive leakage evidence
#       -> LEAKAGE_SUSPECTED
#
# 2. Explicit decision says FAIL / LEAKAGE
#       -> LEAKAGE_SUSPECTED
#
# 3. Explicit decision says PASS / NO_LEAKAGE
#       -> NO_LEAKAGE_EVIDENCE
#
# 4. Explicit negative structural evidence with no positive evidence
#       -> NO_LEAKAGE_EVIDENCE
#
# 5. Otherwise
#       -> INSUFFICIENT_EVIDENCE
#


decision_text = (
    str(decision_value)
    .strip()
    .lower()
    if decision_value is not None
    else ""
)


POSITIVE_DECISION_VALUES = {
    "fail",
    "failed",
    "leakage",
    "leakage_detected",
    "leakage_suspected",
    "suspected",
    "true",
}

NEGATIVE_DECISION_VALUES = {
    "pass",
    "passed",
    "no_leakage",
    "no_leakage_evidence",
    "clean",
    "false",
}


if positive_signals:

    FINAL_LEAKAGE_DECISION = (
        "LEAKAGE_SUSPECTED"
    )

elif decision_text in POSITIVE_DECISION_VALUES:

    FINAL_LEAKAGE_DECISION = (
        "LEAKAGE_SUSPECTED"
    )

elif decision_text in NEGATIVE_DECISION_VALUES:

    FINAL_LEAKAGE_DECISION = (
        "NO_LEAKAGE_EVIDENCE"
    )

elif negative_signals:

    FINAL_LEAKAGE_DECISION = (
        "NO_LEAKAGE_EVIDENCE"
    )

else:

    FINAL_LEAKAGE_DECISION = (
        "INSUFFICIENT_EVIDENCE"
    )


# =============================================================================
# 11. SAFETY GATE
# =============================================================================

if FINAL_LEAKAGE_DECISION == (
    "LEAKAGE_SUSPECTED"
):

    LEAKAGE_GATE = "FAIL"

elif FINAL_LEAKAGE_DECISION == (
    "NO_LEAKAGE_EVIDENCE"
):

    LEAKAGE_GATE = "PASS"

else:

    LEAKAGE_GATE = "WARNING"


print("\n" + "=" * 100)
print("FINAL LEAKAGE DECISION")
print("=" * 100)

print(
    f"Decision : "
    f"{FINAL_LEAKAGE_DECISION}"
)

print(
    f"Leakage gate : "
    f"{LEAKAGE_GATE}"
)

print(
    f"Explicit positive signals : "
    f"{len(positive_signals)}"
)

print(
    f"Explicit negative signals : "
    f"{len(negative_signals)}"
)


# =============================================================================
# 12. ARTIFACT TABLE
# =============================================================================

audit_rows = []

for key, value in flat_payload.items():

    audit_rows.append(
        {
            "field": key,
            "value_type": type(
                value
            ).__name__,
            "value": (
                json.dumps(
                    value,
                    default=str,
                )
                if isinstance(
                    value,
                    (dict, list),
                )
                else str(value)
            ),
        }
    )


audit_df = pd.DataFrame(
    audit_rows
)


CELL9_AUDIT_PATH = (
    CELL9_ROOT
    / "cell9_structural_leakage_field_audit.parquet"
)

audit_df.to_parquet(
    CELL9_AUDIT_PATH,
    index=False,
)


assert CELL9_AUDIT_PATH.exists()


# =============================================================================
# 13. DECISION JSON
# =============================================================================

CELL9_SUMMARY_PATH = (
    CELL9_ROOT
    / "cell9_leakage_decision.json"
)


decision_payload = {

    "cell": 9,

    "source_artifact": str(
        leakage_json_path
    ),

    "final_leakage_decision": (
        FINAL_LEAKAGE_DECISION
    ),

    "leakage_gate": (
        LEAKAGE_GATE
    ),

    "explicit_positive_signal_count": (
        len(positive_signals)
    ),

    "explicit_negative_signal_count": (
        len(negative_signals)
    ),

    "decision_field": (
        decision_key
    ),

    "decision_value": (
        decision_value
    ),

    "direct_leakage_field": (
        direct_key
    ),

    "direct_leakage_value": (
        direct_value
    ),

    "positive_signals": (
        positive_signals
    ),

    "negative_signals": (
        negative_signals
    ),

    "production_inference": (
        "NOT_STARTED"
    ),

    "submission_generation": (
        "NOT_STARTED"
    ),

    "important_note": (
        "Missing leakage evidence is not treated as "
        "proof of no leakage."
    ),
}


with open(
    CELL9_SUMMARY_PATH,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        decision_payload,
        f,
        indent=2,
        default=str,
    )


assert CELL9_SUMMARY_PATH.exists()


# =============================================================================
# 14. FINAL STATUS
# =============================================================================

print("\n" + "=" * 100)
print(
    "TRACE THE ACE — CELL 9 FINAL STATUS"
)
print("=" * 100)

print(
    "Cell 6 structural artifact : PASS"
)

print(
    "Structural field discovery : PASS"
)

print(
    "Explicit leakage signal audit : PASS"
)

print(
    f"FINAL LEAKAGE DECISION : "
    f"{FINAL_LEAKAGE_DECISION}"
)

print(
    f"LEAKAGE GATE : "
    f"{LEAKAGE_GATE}"
)

print(
    "Production inference : NOT STARTED"
)

print(
    "Submission generation : NOT STARTED"
)

print("-" * 100)

print(
    f"Field audit : "
    f"{CELL9_AUDIT_PATH}"
)

print(
    f"Decision JSON : "
    f"{CELL9_SUMMARY_PATH}"
)

print(
    "CELL 9 COMPLETE — PASS"
)

print("=" * 100)

TRACE THE ACE — FINAL MODEL VALIDATION AUDIT
CELL 9 — STRUCTURAL LEAKAGE DECISION RESOLUTION

PROJECT / ARTIFACT CONTRACT
PROJECT_ROOT : D:\Competition\Trace-the-race-local
CELL6_ROOT   : D:\Competition\Trace-the-race-local\scratch_mastery_outputs\09C\final_model_validation_audit\cell6
CELL9_ROOT   : D:\Competition\Trace-the-race-local\scratch_mastery_outputs\09C\final_model_validation_audit\cell9
Project root : PASS
Cell 6 artifact root : PASS

CELL 6 ARTIFACT DISCOVERY
JSON files     : 1
  JSON     : D:\Competition\Trace-the-race-local\scratch_mastery_outputs\09C\final_model_validation_audit\cell6\cell6_structural_leakage_summary.json
Parquet files : 3
  PARQUET  : D:\Competition\Trace-the-race-local\scratch_mastery_outputs\09C\final_model_validation_audit\cell6\cell6_objective_audit.parquet
  PARQUET  : D:\Competition\Trace-the-race-local\scratch_mastery_outputs\09C\final_model_validation_audit\cell6\cell6_objective_fold_audit.parquet
  PARQUET  : D:\Competition\Trace-the-race-local